
### Layered guardrails


Combine multiple guardrails

You can stack multiple guardrails by adding them to the middleware array. They execute in order, allowing you to build layered protection:

In [28]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from rich import print as rprint
import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv()


True

In [12]:
model_free = init_chat_model("openai/gpt-oss-20b",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [13]:
model_advanced

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.6'}}, client=<openrouter.sdk.OpenRouter object at 0x0000023158594050>, openrouter_api_key=SecretStr('**********'), openrouter_api_base='https://openrouter.ai/api/v1', app_url='https://docs.langchain.com', app_title='LangChain', model_name='openai/gpt-5.6-luna-pro', temperature=0.0, max_tokens=10000, model_kwargs={})

In [14]:

# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent, AgentState
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
# --- Middleware building blocks -- every one used in this notebook ---
from langchain.agents.middleware import (
    AgentMiddleware,
    before_model,
    after_model,
    before_agent,
    after_agent,
    wrap_model_call,
    wrap_tool_call,
    hook_config,
    ModelRequest,
    ModelResponse,
)
from typing import Any, Callable
from typing_extensions import NotRequired
from langgraph.runtime import Runtime

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, PIIMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.messages import AIMessage


In [16]:
@tool
def search_tool(query: str) -> str:
    """Search the knowledge base or web for information.
    
    Args:
        query: The search query string.
    """
    return f"Search results for: {query}"


@tool
def send_email(recipient: str, subject: str, body: str, confirm: bool = False) -> str:
    """Send an email to a recipient.
    
    Args:
        recipient: Email address of the recipient.
        subject: Subject line of the email.
        body: Body text of the email message.
        confirm: Confirmation flag required to send the email.
    """
    if not confirm:
        return "Email sending aborted: explicit confirmation required."
    return f"Email successfully sent to {recipient} with subject '{subject}'."


@tool
def delete_database_tool(database_name: str, confirm: bool = False) -> str:
    """Delete a specified database. Use with extreme caution.
    
    Args:
        database_name: Name of the database to delete.
        confirm: Confirmation flag required to proceed.
    """
    if not confirm:
        return "Deletion aborted: confirmation required."
    return f"Database '{database_name}' has been deleted."



In [17]:

class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic guardrail: Block requests containing banned keywords."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        # Check for banned keywords
        for keyword in self.banned_keywords:
            if keyword in content:
                # Block execution before any processing
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
                    }],
                    "jump_to": "end"
                }

        return None


In [18]:

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based guardrail: Use an LLM to evaluate response safety."""

    def __init__(self):
        super().__init__()
        self.safety_model = model_safety

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the final AI response
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a model to evaluate safety
        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content:
            last_message.content = "I cannot provide that response. Please rephrase your request."

        return None

In [36]:

# Create an agent with HITL middleware configured for each tool
agent = create_agent(
    model=model_advanced,
    tools=[search_tool, send_email, delete_database_tool],
    middleware=[
          # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # Layer 2: PII protection (before and after model)
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),

        # Layer 4: Model-based safety check (after agent)
        SafetyGuardrailMiddleware(),
       ],
       checkpointer=InMemorySaver(),
      
  )

In [30]:
graph = agent.get_graph()

print(graph.draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	ContentFilterMiddleware\2ebefore_agent(ContentFilterMiddleware.before_agent)
	PIIMiddleware\5bemail\5d\2ebefore_model(PIIMiddleware[email].before_model)
	PIIMiddleware\5bemail\5d\2eafter_model(PIIMiddleware[email].after_model)
	HumanInTheLoopMiddleware\2eafter_model(HumanInTheLoopMiddleware.after_model)
	SafetyGuardrailMiddleware\2eafter_agent(SafetyGuardrailMiddleware.after_agent)
	__end__([<p>__end__</p>]):::last
	ContentFilterMiddleware\2ebefore_agent -.-> PIIMiddleware\5bemail\5d\2ebefore_model;
	ContentFilterMiddleware\2ebefore_agent -.-> SafetyGuardrailMiddleware\2eafter_agent;
	HumanInTheLoopMiddleware\2eafter_model --> PIIMiddleware\5bemail\5d\2eafter_model;
	PIIMiddleware\5bemail\5d\2eafter_model -.-> PIIMiddleware\5bemail\5d\2ebefore_model;
	PIIMiddleware\5bemail\5d\2eafter_model -.-> SafetyGuardrailMiddleware\2eafter_agent;
	PIIMiddleware\5bemail\5d\2e

In [31]:

# 1. Trigger Layer 1 (Content Filter): Banned keyword
print("--- Triggering Layer 1 ---")
result_layer1 = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack the mainframe?"}]
})
rprint(result_layer1["messages"][-1].content)


--- Triggering Layer 1 ---


I cannot process requests containing inappropriate content. Please rephrase your request.

In [32]:

# 2. Trigger Layer 2 (PII Protection): Input with email address
print("\n--- Triggering Layer 2 ---")
result_layer2 = agent.invoke({
    "messages": [{"role": "user", "content": "Find information for user test@example.com"}]
})
rprint(result_layer2["messages"][-1].content)



--- Triggering Layer 2 ---


What information are you looking for—such as a public professional profile, account status, or contact details?

I can help search for **publicly available information**, but I can’t retrieve private account data, passwords, 
messages, or other sensitive personal information associated with an email address.

In [37]:
import uuid
thread_id= str(uuid)
# Human-in-the-loop requires a thread ID for persistence
config = {"configurable": {"thread_id": thread_id}}

# 3. Trigger Layer 3 (Human-in-the-Loop): Calling sensitive tool
print("\n--- Triggering Layer 3 ---")
result_layer3 = agent.invoke({
    "messages": [{"role": "user", "content": "Send an email to user@example.com saying hello."}]
}, 
config=config,)
rprint(result_layer3)


--- Triggering Layer 3 ---


{
    'messages': [
        HumanMessage(
            content='Send an email to [REDACTED_EMAIL] saying hello.',
            additional_kwargs={},
            response_metadata={},
            id='fd13c7a2-b994-49aa-806b-f891317e190b'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': '**Confirming tool usage**\n\nI’m thinking about whether I need confirmation 
to use the tool. The user did say to send, so it seems like confirmation should be set to true because of their 
explicit instruction. The message says "confirm required to send," which hints at a tool-specific safety measure. 
Since the user’s request was clear, I can go ahead and set the confirmation flag to true. I’ll include a friendly 
greeting in the message body.',
                'reasoning_details': [
                    {
                        'summary': '**Confirming tool usage**\n\nI’m thinking about whether I need confirmation to 
use the tool. The user did say to send, so it seems like confirmation should be set to true because of their 
explicit instruction. The message says "confirm required to send," which hints at a tool-specific safety measure. 
Since the user’s request was clear, I can go ahead and set the confirmation flag to true. I’ll include a friendly 
greeting in the message body.',
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqhRkjTEDhh2AuKRnK3nXNbXkNz_EVsv09BkyjQ9yXjURuSPcfFnVq66l1ZThseL9flAs2bs_JQySMJDHkEX1uVI8IX3nTOzDGmMsaLKmhMe
FbRNsI_8w1RrefIl_nwABBUMhiocas9ebwpGzbvcQV3zDT7TZropDFk6IRZX6nIByL4FNbs3umOidZ9QhyVD4_ElcwsD-KGw3h3aIo1NVHVCTtLrDn-
hz93vWWYzMwfKSsapzDs_rG0pGDBA3pErSMsNSKr_oPmrG9pruLR5b5wv8aduCkZwD3FzHZtNqh1k0w0LjduWtPcFh3npmyoANjRDzlJ8ZUbEEifqq7
5xtQbdtoAsceKregzBlRcr4QCqTl-sXmFjoi1RXtae6lIPLeilF2zj6D1RICVUlqEG-3UJR8yWZ-aSBycqI55odsQjROX00kApnJDCinx2WzNTCfg0E
Ly6GUpq_VZqUOK8F5P3egR7aG4iq55H2BxeZQohAPmOpV3KP-eR2G8MOh4VrsTA12yP2Q1PPjsC8sBDGSxHNM5HQEUuHMkmpxyiF1VQWUB4mKCdwO_7
IYCUerdcDv_2VOG0sAMaxmSndTdk5bzHYJxnmv2KdyB0C4tXJGymeM4qWU5P7jRiftl6PkJvYd6DrjxY2IZjOmvy3zh0VHPWNi5ilypYb_Qpu9zeuTn
9x_crMG-buw4UfN8PfVqdG237Fv6NqIAMWf4sbveoihynxcTLpb5lGPRn-bWQ5Y8BKWTc81Z0f4VhFCpVR7KZ2I10y9k85nUUa_Fxaa5rjh3TmaVa_X
aPk3pCKmImA0s28FoCzxNWbLaRNDjOmBQzhAXbdGd_c7TYWsvrufjINuCs5ESh-_fq8xYq33Bdq5jo4-9HmraGDrPLNzcNGGYSxQjf6RUD2KkhegY2U
vwkb9G9c5PYoDK7ZxMHizHDC3scVWpB4DJ0n0KUxNT-2ZB8GcSdWn9FSy4So7huf6KHofLi-2P83b1Ux9JyNG6QqWkFoO_1o0lYdy_tlgtVyoKkgFFq
ZjWIThvcWWC1Nq_jQrn1jUxeDetQz19O39-R9WGSCbGw9LB-6Dv4yFSz7nAqsSlrKaNorUpgS_nHlk98MtIl09QDwmQWX0mdpyJnZZxn_GOgeL2fXqG
9cGHkkKf2H3fdidBn_UWnzT7wN7b5jF3GndannHglIS3vy17APguSHgLYKnpxRPqMzAqI_IWFhoOQR6vMoYzxbjHoIEcm70RVDSnJQu3eU4-JziVaNJ
nlIschEuH89DwS2jHddYt0LeqjoJ_fUznpyAIk7ltYjyfHZdzTald-SLZhvP4HHU14CmMXIrSpUaNcGUmUI1G0VeJF852GE4yPYYXu35bnSz0ZUTlUA
YjQ0kTdYyd4tfyopHBzN0n9s9tZ3GsdHGbBCTXMPGFEPk77dfXp-gdfM81d1pDCGw3mb0l_S96zRBNF9u83nVlWTUZU9WWvMXvr9UkQpFqdKi6qnldX
DhTC_MEGige4Kwn2tOZqqQASD1V62ePaTrD5cFMN4Jo5jUmSlEl1dA7oI3J79UhypEAs2X47TquadmC3wnCfSenns65OmJGS0nt-TxAKzdEo13GlvJU
ahAqJ1qprchn4YXyOg5vDtwVS-2SaWRK7cnUwE-mjgkqjAGfd4CI5DvMWGqffibUrSq55fxcEZyPC3rRevf2sQHLxGvwHpaFoNe3etjkdnRBZfWl3ov
afPRudxRBie-aBGXct9tPXaD6r6Sbl6udsotzNDgPFB93O0NNeBEYV2VU4KwXYHM-rupVaNlWNZc8XL6.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2
dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0d6fadf6a2ce3156016a85192198f087d18718bce688f34154',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1787107611-64ihvJ9gTa0gxuaSFVJm',
                'created': 1787107611,
                'object': 'chat.com

In [38]:
print("\n--- Triggering HITL response ---")
resultHITL = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread ID to resume the paused conversation
)

rprint(resultHITL)


--- Triggering HITL response ---


{
    'messages': [
        HumanMessage(
            content='Send an email to [REDACTED_EMAIL] saying hello.',
            additional_kwargs={},
            response_metadata={},
            id='fd13c7a2-b994-49aa-806b-f891317e190b'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': '**Confirming tool usage**\n\nI’m thinking about whether I need confirmation 
to use the tool. The user did say to send, so it seems like confirmation should be set to true because of their 
explicit instruction. The message says "confirm required to send," which hints at a tool-specific safety measure. 
Since the user’s request was clear, I can go ahead and set the confirmation flag to true. I’ll include a friendly 
greeting in the message body.',
                'reasoning_details': [
                    {
                        'summary': '**Confirming tool usage**\n\nI’m thinking about whether I need confirmation to 
use the tool. The user did say to send, so it seems like confirmation should be set to true because of their 
explicit instruction. The message says "confirm required to send," which hints at a tool-specific safety measure. 
Since the user’s request was clear, I can go ahead and set the confirmation flag to true. I’ll include a friendly 
greeting in the message body.',
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqhRkjTEDhh2AuKRnK3nXNbXkNz_EVsv09BkyjQ9yXjURuSPcfFnVq66l1ZThseL9flAs2bs_JQySMJDHkEX1uVI8IX3nTOzDGmMsaLKmhMe
FbRNsI_8w1RrefIl_nwABBUMhiocas9ebwpGzbvcQV3zDT7TZropDFk6IRZX6nIByL4FNbs3umOidZ9QhyVD4_ElcwsD-KGw3h3aIo1NVHVCTtLrDn-
hz93vWWYzMwfKSsapzDs_rG0pGDBA3pErSMsNSKr_oPmrG9pruLR5b5wv8aduCkZwD3FzHZtNqh1k0w0LjduWtPcFh3npmyoANjRDzlJ8ZUbEEifqq7
5xtQbdtoAsceKregzBlRcr4QCqTl-sXmFjoi1RXtae6lIPLeilF2zj6D1RICVUlqEG-3UJR8yWZ-aSBycqI55odsQjROX00kApnJDCinx2WzNTCfg0E
Ly6GUpq_VZqUOK8F5P3egR7aG4iq55H2BxeZQohAPmOpV3KP-eR2G8MOh4VrsTA12yP2Q1PPjsC8sBDGSxHNM5HQEUuHMkmpxyiF1VQWUB4mKCdwO_7
IYCUerdcDv_2VOG0sAMaxmSndTdk5bzHYJxnmv2KdyB0C4tXJGymeM4qWU5P7jRiftl6PkJvYd6DrjxY2IZjOmvy3zh0VHPWNi5ilypYb_Qpu9zeuTn
9x_crMG-buw4UfN8PfVqdG237Fv6NqIAMWf4sbveoihynxcTLpb5lGPRn-bWQ5Y8BKWTc81Z0f4VhFCpVR7KZ2I10y9k85nUUa_Fxaa5rjh3TmaVa_X
aPk3pCKmImA0s28FoCzxNWbLaRNDjOmBQzhAXbdGd_c7TYWsvrufjINuCs5ESh-_fq8xYq33Bdq5jo4-9HmraGDrPLNzcNGGYSxQjf6RUD2KkhegY2U
vwkb9G9c5PYoDK7ZxMHizHDC3scVWpB4DJ0n0KUxNT-2ZB8GcSdWn9FSy4So7huf6KHofLi-2P83b1Ux9JyNG6QqWkFoO_1o0lYdy_tlgtVyoKkgFFq
ZjWIThvcWWC1Nq_jQrn1jUxeDetQz19O39-R9WGSCbGw9LB-6Dv4yFSz7nAqsSlrKaNorUpgS_nHlk98MtIl09QDwmQWX0mdpyJnZZxn_GOgeL2fXqG
9cGHkkKf2H3fdidBn_UWnzT7wN7b5jF3GndannHglIS3vy17APguSHgLYKnpxRPqMzAqI_IWFhoOQR6vMoYzxbjHoIEcm70RVDSnJQu3eU4-JziVaNJ
nlIschEuH89DwS2jHddYt0LeqjoJ_fUznpyAIk7ltYjyfHZdzTald-SLZhvP4HHU14CmMXIrSpUaNcGUmUI1G0VeJF852GE4yPYYXu35bnSz0ZUTlUA
YjQ0kTdYyd4tfyopHBzN0n9s9tZ3GsdHGbBCTXMPGFEPk77dfXp-gdfM81d1pDCGw3mb0l_S96zRBNF9u83nVlWTUZU9WWvMXvr9UkQpFqdKi6qnldX
DhTC_MEGige4Kwn2tOZqqQASD1V62ePaTrD5cFMN4Jo5jUmSlEl1dA7oI3J79UhypEAs2X47TquadmC3wnCfSenns65OmJGS0nt-TxAKzdEo13GlvJU
ahAqJ1qprchn4YXyOg5vDtwVS-2SaWRK7cnUwE-mjgkqjAGfd4CI5DvMWGqffibUrSq55fxcEZyPC3rRevf2sQHLxGvwHpaFoNe3etjkdnRBZfWl3ov
afPRudxRBie-aBGXct9tPXaD6r6Sbl6udsotzNDgPFB93O0NNeBEYV2VU4KwXYHM-rupVaNlWNZc8XL6.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2
dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0d6fadf6a2ce3156016a85192198f087d18718bce688f34154',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1787107611-64ihvJ9gTa0gxuaSFVJm',
                'created': 1787107611,
                'object': 'chat.com